<a href="https://colab.research.google.com/github/LeleGll/SML-2026/blob/main/Diffusion_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diffusion Exercise: Swiss Roll in 2D

Create a copy of this notebook File -> Save a Copy in Drive (or Download) to be able to save your progress.

This notebook is a guided coding exercise where you build diffusion components in two stages:
1. a naive residual denoiser that learns to undo small noising steps
2. a DDPM-style model that predicts the Gaussian noise `eps`

## Learning goals
1. Understand the difference between one-step noising and closed-form noising.
2. Train a simple timestep-conditioned denoiser on 2D data.
3. Implement the standard DDPM training pattern: sample `t`, sample `eps`, construct `x_t`, predict `eps`, minimize MSE.
4. Use plots and widgets to inspect what the model learned at different timesteps.

## Main reference
- Ho, Jain, Abbeel (2020), *Denoising Diffusion Probabilistic Models* (NeurIPS 2020): https://arxiv.org/abs/2006.11239

## What you need to fill in
- `Ex1`: complete the MLP.
- `Ex2`: implement the toy forward noising step.
- `Toy warmup`: implement one training step for the naive denoiser.
- `Ex3`: implement DDPM-style noising and the reverse mean update.
- `Ex4`: implement the DDPM training loss.

The toy model is intentionally simpler than a full DDPM. It helps you see the denoising idea before switching to epsilon prediction.


## Setup and imports

Run this once. In Colab, uncomment the install line if needed.


In [ ]:
# Uncomment in a fresh Colab runtime if needed:
# %pip install -q numpy matplotlib ipywidgets scikit-learn torch

import math
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import ipywidgets as widgets
from IPython.display import display

from sklearn.datasets import make_swiss_roll


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Data and plotting helpers


In [ ]:
class SwissRoll2D(Dataset):
    """
    Create a 2D Swiss roll by taking (x, z) coordinates from the 3D generator.
    Optionally normalize to mean 0 and variance 1 per axis.
    """

    def __init__(self, num_samples=10000, noise=0.05, normalize=True):
        data3d, _ = make_swiss_roll(n_samples=num_samples, noise=noise)
        xz = data3d[:, [0, 2]].astype(np.float32)
        x = torch.from_numpy(xz)
        if normalize:
            mu = x.mean(dim=0, keepdim=True)
            std = x.std(dim=0, keepdim=True).clamp_min(1e-6)
            x = (x - mu) / std
        self.data = x

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]


def data_limits(x, pad=0.1):
    x_np = x.detach().cpu().numpy()
    xmin, ymin = x_np.min(axis=0)
    xmax, ymax = x_np.max(axis=0)
    xr = xmax - xmin
    yr = ymax - ymin
    return xmin - pad * xr, xmax + pad * xr, ymin - pad * yr, ymax + pad * yr


def plot_points(x, title='Points', s=5, limits=None, color='tab:blue'):
    x_np = x.detach().cpu().numpy()
    if limits is None:
        limits = data_limits(x)
    xmin, xmax, ymin, ymax = limits
    plt.figure(figsize=(5, 5))
    plt.scatter(x_np[:, 0], x_np[:, 1], s=s, alpha=0.65, c=color)
    plt.title(title)
    ax = plt.gca()
    ax.set_aspect('equal', 'box')
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)
    plt.grid(alpha=0.2)
    plt.show()


def t_to_norm(t: torch.Tensor, T: int) -> torch.Tensor:
    if t.dim() == 0:
        t = t.view(1)
    return (t.float() / max(T - 1, 1)).unsqueeze(1)


In [ ]:
BATCH_SIZE = 256
dataset = SwissRoll2D(num_samples=10000, noise=0.05, normalize=True)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

with torch.no_grad():
    idx = torch.randperm(len(dataset))[:2500]
    plot_points(dataset.data[idx], title='Normalized Swiss Roll (2D)')


## One-step, n-step, and naive denoiser

Let $x_0$ be a clean data point, and $x_t$ the noised version at timestep $t$.

Define
$$
\alpha = 1-\beta
$$
so the one-step noising rule becomes
$$
x_{t+1} = \alpha x_t + (1-\alpha)\,\epsilon_t,\quad \epsilon_t \sim \mathcal{N}(0, I).
$$

### Exercise: derive the n-step forward expression
1. Expand the recursion for $x_1$, $x_2$, $x_3$.
2. Show the intermediate sum form
$$
x_t = \alpha^t x_0 + (1-\alpha)\sum_{k=0}^{t-1}\alpha^k\epsilon_{t-1-k}.
$$
3. Use the fact that a linear combination of independent Gaussians is again Gaussian to rewrite the noise term as a single Gaussian.

If we introduce
$$
\bar\alpha_t = \prod_{s=1}^t \alpha_s,
$$
then in this toy constant-$\alpha$ case we simply have $\bar\alpha_t = \alpha^t$, and the final compact form is
$$
x_t = \bar\alpha_t x_0 + \sqrt{1-\bar\alpha_t^2}\,z,\quad z \sim \mathcal{N}(0, I).
$$

This is the same idea as the usual DDPM notation: the whole multi-step forward process can be written as one Gaussian corruption of $x_0$.

### 3) What the naive denoiser learns
In this warmup, the network takes $(x_t, t)$ and predicts a denoised point directly.
You can think of it as learning a map from a noisy point toward a cleaner one-step predecessor.


## Exercise 1: Build the denoiser MLP

Implement the missing layers marked with `TODO`.

Requirements:
- input dim = 3 because we concatenate 2D coordinates with normalized time `t_norm`
- output dim = 2 because the network predicts a 2D denoised point
- use at least two hidden layers with nonlinear activations

Recommended structure:
`Linear(3, hidden) -> activation -> Linear(hidden, hidden) -> activation -> Linear(hidden, 2)`


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden),
            # TODO(Ex1): add nonlinear activation
            # TODO(Ex1): add a second hidden linear layer
            # TODO(Ex1): add nonlinear activation
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t_norm):
        h = torch.cat([x, t_norm], dim=1)
        return self.net(h)


## Exercise 2: Simple diffusion forward step

Use the toy one-step process
$$
x_{t+1} = \alpha x_t + (1-\alpha)\,\epsilon,\quad \epsilon \sim \mathcal{N}(0, I),\qquad \alpha = 1-\beta
$$

Your job in `step_forward` is just to translate that equation into code:
- keep an `alpha = 1 - beta` fraction of the current point
- add fresh Gaussian noise with weight `(1 - alpha) = beta`
- return the resulting `x_next`

This toy process is the forward model used in the naive denoiser warmup below.


In [ ]:
class SimpleDiffusion(nn.Module):
    def __init__(self, T=10, beta=0.12, hidden=64, device=None):
        super().__init__()
        self.T = int(T)
        self.beta = float(beta)
        self.device = device if device is not None else torch.device('cpu')
        self.net = MLP(hidden=hidden)

    def forward(self, x_t, t):
        if t.dim() == 0:
            t = t.expand(x_t.shape[0])
        t_norm = t_to_norm(t, self.T).to(x_t.device)
        return self.net(x_t, t_norm)

    @torch.no_grad()
    def step_forward(self, x):
        noise = torch.randn_like(x)
        # TODO(Ex2): implement x_next = (1 - beta) * x + beta * noise
        raise NotImplementedError('Ex2: implement SimpleDiffusion.step_forward')

    @torch.no_grad()
    def q_sample(self, x0, t, eps=None):
        # Iterative noising for toy diffusion
        steps = int(t[0].item()) if t.dim() > 0 else int(t.item())
        x = x0.clone()
        for _ in range(steps):
            x = self.step_forward(x)
        return x

    @torch.no_grad()
    def sample(self, n, step_size=1.0, eta=0.0, collect_ts=None):
        x = torch.randn(n, 2, device=self.device)
        snaps = {}
        for t_int in reversed(range(self.T)):
            t_vec = torch.full((n,), t_int, device=self.device, dtype=torch.long)
            x_prev_pred = self.forward(x, t_vec)
            x = (1.0 - step_size) * x + step_size * x_prev_pred
            if collect_ts is not None and t_int in collect_ts:
                snaps[t_int] = x.detach().clone()
        snaps[0] = x.detach().clone()
        return x, snaps


## Guided warmup: naive denoiser training step

Before training the DDPM, implement one training step for the toy denoiser.

What should happen in `toy_train_step`:
1. build a short forward trajectory `x_0, x_1, ..., x_T` using `step_forward`
2. for each `t` from `T` down to `1`, feed `(x_t, t-1)` into the network
3. interpret the network output as a prediction of the cleaner point `x_{t-1}`
4. compare that prediction to the true `x_{t-1}` with MSE
5. average the losses over timesteps

This warmup is deliberately explicit. It is slower than vectorized DDPM training, but easier to understand.


In [ ]:
def toy_train_step(model, x0):
    """
    Build one batch loss for the naive denoiser.
    The model predicts the cleaner point x_{t-1} from (x_t, t).
    """
    bsz = x0.size(0)

    xts = [x0]
    for _ in range(model.T):
        xts.append(model.step_forward(xts[-1]))

    total_loss = 0.0
    for t in reversed(range(1, model.T + 1)):
        xt = xts[t]
        x_prev = xts[t - 1]

        t_vec = torch.full((bsz,), t - 1, device=x0.device, dtype=torch.long)

        # TODO(Toy warmup): predict the cleaner point x_{t-1} from x_t
        # x_prev_pred = ...

        # TODO(Toy warmup): compare prediction to the true x_{t-1}
        # loss_t = ...

        total_loss = total_loss + loss_t

    return total_loss / model.T


## Naive denoiser: model setup, training, and inspection

Complete this whole section before moving to DDPM.
The point is to see one full pipeline first: define the toy forward process, train a residual denoiser, and inspect what it learned.


In [ ]:
toy_model = SimpleDiffusion(T=10, beta=0.12, hidden=64, device=device).to(device)
print('Naive model params:', sum(p.numel() for p in toy_model.parameters() if p.requires_grad))


### Optional warmup training loop for the naive denoiser

This loop is intentionally explicit.
A very small training loss is **not** the main success criterion here: the toy problem is low-dimensional and the one-step denoising target can become easy to fit.
Use the reverse samples and vector field as the real check that the implementation makes sense.


In [ ]:
TOY_EPOCHS = 200

toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)
toy_losses = []

toy_model.train()
for epoch in range(1, TOY_EPOCHS + 1):
    running, batches = 0.0, 0

    for x0 in loader:
        x0 = x0.to(device)
        loss = toy_train_step(toy_model, x0)

        toy_optimizer.zero_grad()
        loss.backward()
        toy_optimizer.step()

        running += loss.item()
        batches += 1

    avg = running / max(batches, 1)
    toy_losses.append(avg)
    if epoch % 25 == 0 or epoch == 1:
        print(f'[Toy] Epoch {epoch:3d}/{TOY_EPOCHS} | loss: {avg:.5f}')

plt.figure(figsize=(6, 4))
plt.plot(toy_losses)
plt.title('Naive denoiser train loss')
plt.xlabel('epoch')
plt.ylabel('MSE')
plt.grid(alpha=0.3)
plt.show()


## Interactive visualization helpers for the naive denoiser

These widgets let you inspect the toy denoiser:
1. forward diffusion at a selected timestep
2. reverse denoising snapshots
3. timestep-conditioned denoising vector field


In [ ]:
@torch.no_grad()
def make_forward_widget(model, x0, n_show=2500):
    x0 = x0.to(model.device)
    idx = torch.randperm(x0.size(0), device=model.device)[:min(n_show, x0.size(0))]
    x0_sub = x0[idx]
    limits = data_limits(x0_sub)

    def _plot(t):
        t = int(t)
        t_vec = torch.full((x0_sub.size(0),), t, device=model.device, dtype=torch.long)
        x_t = model.q_sample(x0_sub, t_vec)
        plot_points(x_t.cpu(), title=f'Forward process at t={t}', limits=limits)

    return widgets.interact(
        _plot,
        t=widgets.IntSlider(value=0, min=0, max=model.T - 1, step=1, description='t')
    )


@torch.no_grad()
def make_reverse_widget(model, ref_points, n_show=3000, step_size=0.8, eta=0.6):
    ref_points = ref_points.to(model.device)
    limits = data_limits(ref_points)
    ts = list(range(model.T))
    _, snaps = model.sample(n=n_show, step_size=step_size, eta=eta, collect_ts=ts)

    def _plot(t):
        t = int(t)
        pts = snaps[t]
        plot_points(pts.cpu(), title=f'Reverse samples at t={t}', limits=limits, color='tab:orange')

    return widgets.interact(
        _plot,
        t=widgets.IntSlider(value=model.T - 1, min=0, max=model.T - 1, step=1, description='t')
    )


@torch.no_grad()
def _denoise_direction(model, points, t_int):
    t_vec = torch.full((points.size(0),), int(t_int), device=model.device, dtype=torch.long)
    if hasattr(model, 'alpha_bars') and hasattr(model, 'betas'):
        eps_pred = model(points, t_vec)
        beta_t = model.betas[t_vec].unsqueeze(1)
        alpha_t = model.alphas[t_vec].unsqueeze(1)
        alpha_bar_t = model.alpha_bars[t_vec].unsqueeze(1)
        mean_step = (points - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred) / torch.sqrt(alpha_t)
        return mean_step - points
    x_prev_pred = model(points, t_vec)
    return x_prev_pred - points


@torch.no_grad()
def make_vector_field_widget(model, ref_points, grid_size=20, n_overlay=1200):
    ref_points = ref_points.to(model.device)
    limits = data_limits(ref_points)
    xmin, xmax, ymin, ymax = limits

    xs = torch.linspace(xmin, xmax, grid_size, device=model.device)
    ys = torch.linspace(ymin, ymax, grid_size, device=model.device)
    grid = torch.cartesian_prod(xs, ys)

    idx = torch.randperm(ref_points.size(0), device=model.device)[:min(n_overlay, ref_points.size(0))]
    x0_overlay = ref_points[idx]

    def _plot(t):
        t = int(t)
        t_overlay = torch.full((x0_overlay.size(0),), t, device=model.device, dtype=torch.long)
        x_t_overlay = model.q_sample(x0_overlay, t_overlay)

        v = _denoise_direction(model, grid, t).cpu().numpy()
        g = grid.cpu().numpy()
        norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-8
        v = v / norm * 2

        x_np = x_t_overlay.cpu().numpy()
        plt.figure(figsize=(6, 6))
        plt.scatter(x_np[:, 0], x_np[:, 1], s=5, alpha=0.25, c='tab:blue', label='x_t cloud')
        plt.quiver(
            g[:, 0], g[:, 1], v[:, 0], v[:, 1],
            angles='xy', scale_units='xy', scale=20,
            width=0.003, alpha=0.9, color='tab:red'
        )
        plt.title(f'Denoising vector field conditioned on t={t}')
        ax = plt.gca()
        ax.set_aspect('equal', 'box')
        plt.xlim(xmin, xmax)
        plt.ylim(ymin, ymax)
        plt.grid(alpha=0.2)
        plt.legend(loc='upper right')
        plt.show()

    return widgets.interact(
        _plot,
        t=widgets.IntSlider(value=model.T // 2, min=0, max=model.T - 1, step=1, description='t')
    )


In [ ]:
toy_model.eval()

print('Naive forward explorer')
make_forward_widget(toy_model, dataset.data, n_show=2500)

print('Naive reverse explorer')
make_reverse_widget(toy_model, dataset.data, n_show=3000, step_size=1.0, eta=0.0)

print('Naive vector field explorer')
make_vector_field_widget(toy_model, dataset.data, grid_size=19, n_overlay=1400)


## Why move beyond direct denoised-point prediction

The naive model predicts a cleaner point directly from `x_t`. That is a useful warmup, but for real diffusion models it is usually not the best parameterization.

Why direct clean-point prediction is awkward:
- the prediction target changes a lot with timestep: early `t` is almost clean, late `t` is dominated by noise
- one network must handle many different target scales and geometries
- optimization is often better behaved when the model predicts Gaussian noise `eps`, whose distribution is much more uniform across timesteps

This motivates the DDPM parameterization below: instead of directly predicting a denoised point, the network predicts the noise that was added.


## Exercise 3: DDPM core methods

Now move from the naive residual denoiser to a DDPM-style epsilon predictor.

Use the closed-form forward process from DDPMs rather than simulating the noising chain step by step:
$$
x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)
$$
$$
\beta_t \in (0,1), \qquad \alpha_t = 1 - \beta_t, \qquad \bar\alpha_t = \prod_{s=1}^t \alpha_s.
$$

This is what is typically done in practice during training: for each example in the batch, sample a timestep $t$, sample Gaussian noise $\epsilon$, and construct $x_t$ directly from $x_0$ with the closed form.

In this exercise:
- `q_sample` should build $x_t = \sqrt{\bar\alpha_t} \, x_0 + \sqrt{1 - \bar\alpha_t} \, \epsilon$
- `sample` should use the predicted noise in a DDPM-style reverse mean update


In [ ]:
class DDPM(nn.Module):
    def __init__(self, T=100, beta_start=1e-4, beta_end=2e-2, hidden=128, device=None):
        super().__init__()
        self.T = int(T)
        self.device = device if device is not None else torch.device('cpu')
        self.net = MLP(hidden=hidden)

        betas = torch.linspace(beta_start, beta_end, self.T)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_bars', alpha_bars)

    def alpha_bar(self, t):
        if t.dim() == 0:
            t = t.view(1)
        return self.alpha_bars[t].unsqueeze(1)

    def forward(self, x_t, t):
        if t.dim() == 0:
            t = t.expand(x_t.shape[0])
        t_norm = t_to_norm(t, self.T).to(x_t.device)
        return self.net(x_t, t_norm)

    @torch.no_grad()
    def q_sample(self, x0, t, eps=None):
        if eps is None:
            eps = torch.randn_like(x0)
        alpha_bar_t = self.alpha_bar(t).to(x0.device)
        # TODO(Ex3): return sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps
        raise NotImplementedError('Ex3: implement DDPM.q_sample')

    @torch.no_grad()
    def sample(self, n, step_size=0.8, eta=0.6, collect_ts=None):
        x = torch.randn(n, 2, device=self.device)
        snaps = {}

        for t_int in reversed(range(self.T)):
            t_vec = torch.full((n,), t_int, device=self.device, dtype=torch.long)
            eps_pred = self.forward(x, t_vec)
            beta_t = self.betas[t_vec].unsqueeze(1)
            alpha_t = self.alphas[t_vec].unsqueeze(1)
            alpha_bar_t = self.alpha_bars[t_vec].unsqueeze(1)
            # TODO(Ex3): implement the DDPM reverse mean update
            # mean_step = (x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred) / torch.sqrt(alpha_t)
            raise NotImplementedError('Ex3: implement DDPM.sample mean_step update')

            if eta > 0.0 and t_int > 0:
                z = torch.randn_like(x)
                alpha_bar_prev = self.alpha_bars[t_int - 1]
                posterior_var = beta_t * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)
                x = mean_step + eta * torch.sqrt(posterior_var) * z
            else:
                x = mean_step

            if collect_ts is not None and t_int in collect_ts:
                snaps[t_int] = x.detach().clone()

        snaps[0] = x.detach().clone()
        return x, snaps


## Exercise 4: Timestep sampling and DDPM training loss

Implement both TODO blocks.

For each training batch, the standard pattern is:
1. sample one timestep $t$ independently for each example
2. sample Gaussian noise $\epsilon \sim \mathcal{N}(0, I)$ with the same shape as $x_0$
3. create the noisy input with the closed form
$$
x_t = \sqrt{\bar\alpha_t} \, x_0 + \sqrt{1 - \bar\alpha_t} \, \epsilon
$$
4. run the model on $(x_t, t)$ to predict $\epsilon$
5. return `F.mse_loss(eps_pred, eps)`

This is the core DDPM training recipe used in practice.


In [ ]:
def sample_timesteps(batch_size, T, device):
    # TODO(Ex4): sample one integer timestep per example in [0, T)
    # expected shape: (batch_size,)
    raise NotImplementedError('Ex4: implement sample_timesteps')


def ddpm_train_step(model, x0):
    """
    Build noised inputs and return MSE loss for epsilon prediction.
    """
    bsz = x0.size(0)
    t = sample_timesteps(batch_size=bsz, T=model.T, device=x0.device)
    eps = torch.randn_like(x0)
    x_t = model.q_sample(x0, t, eps)
    eps_pred = model(x_t, t)

    # TODO(Ex4): return MSE(eps_pred, eps)
    raise NotImplementedError('Ex4: implement ddpm_train_step loss')


## DDPM: model setup and training

Now switch to the main model in this notebook: epsilon prediction with closed-form forward noising.


In [ ]:
model = DDPM(T=100, beta_start=1e-4, beta_end=2e-2, hidden=128, device=device).to(device)
print('DDPM params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


### Main DDPM training loop

This is the standard vectorized epsilon-prediction loop.
For each batch, sample one random timestep per example, construct $x_t$ directly from the closed form, predict $\epsilon$, and minimize MSE.


In [ ]:
EPOCHS = 400
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

model.train()
for epoch in range(1, EPOCHS + 1):
    running, batches = 0.0, 0

    for x0 in loader:
        x0 = x0.to(device)
        loss = ddpm_train_step(model, x0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item()
        batches += 1

    avg = running / max(batches, 1)
    losses.append(avg)
    if epoch % 50 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | loss: {avg:.5f}')

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title('DDPM train loss')
plt.xlabel('epoch')
plt.ylabel('MSE')
plt.grid(alpha=0.3)
plt.show()


## Interactive visualization helpers for DDPM

These widgets let you inspect the trained DDPM:
1. forward diffusion at a selected timestep
2. reverse denoising snapshots
3. timestep-conditioned denoising vector field


In [ ]:
model.eval()

print('Forward explorer')
make_forward_widget(model, dataset.data, n_show=2500)

print('Reverse explorer')
make_reverse_widget(model, dataset.data, n_show=3000, step_size=0.8, eta=0.6)

print('Vector field explorer')
make_vector_field_widget(model, dataset.data, grid_size=19, n_overlay=1400)


## Optional extension tasks

1. Try different beta schedules `beta_t` and inspect how they change `\bar\alpha_t`.
2. Compare deterministic sampling (`eta=0`) vs stochastic sampling (`eta>0`).
3. Increase/decrease `T` and explain the effect on generated samples.
